In [1]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
# from langchain_core.runnables import RunnableConfig
from langchain_openai import ChatOpenAI
# from langgraph.graph import END, START, StateGraph
from typing_extensions import TypedDict, Literal
import json, random
from tqdm import tqdm
import os

from dotenv import load_dotenv
load_dotenv()


True

# Load Fold 5 Datasets

In [2]:
# Setting file paths
fold_path = os.path.join(os.path.dirname(os.getcwd()), 'data', 'pqal_fold3')
train_data = json.load(open(os.path.join(fold_path, "train_set.json")))
dev_data = json.load(open(os.path.join(fold_path, "dev_set.json")))

print(f"# Train examples: {len(train_data)}")
print(f"# Dev examples: {len(dev_data)}")


# Train examples: 450
# Dev examples: 50


# Select Few-Shot Examples from Train Set

In [10]:
# # import random

# few_shot_examples = [
#     {
#         "context": "Statins are used to treat cardiovascular disease. This study evaluates their effectiveness in older adults.",
#         "question": "Are statins effective in elderly patients for reducing heart attacks?",
#         "rationale": "The study population includes older adults and finds a statistically significant reduction in heart attacks among those taking statins.",
#         "answer": "yes"
#     },
#     {
#         "context": "A small trial looked at a new treatment for Alzheimer's disease. The results were mixed and inconclusive.",
#         "question": "Does the new treatment improve memory in Alzheimer's patients?",
#         "rationale": "While some patients improved, the sample size was small and the results were not statistically significant.",
#         "answer": "maybe"
#     },

#     {
#     "context": (
#         "Urine samples were examined by wet smear microscopy, incubated in 5% CO₂ for 1–2 days, and species-specific PCR for A. schaalii was performed. "
#         "In 5 of the 29 screened urines, A. schaalii was found only by PCR in moderate quantities. In two cases, both PCR and culture found A. schaalii with signs of infection. "
#         "The authors recommend PCR or Gram staining when cultures are negative but symptoms persist."
#     ),
#     "question": "Actinobaculum schaalii, a cause of urinary tract infections in children?",
#     "rationale": (
#         "The study shows A. schaalii was found in some urine samples, mostly via PCR, and only a few cases had both PCR and culture evidence with symptoms. "
#         "The authors suggest testing when symptoms are present but standard culture is negative, indicating possible relevance. "
#         "However, the evidence is limited and based on a small number of cases, so it is suggestive but not conclusive."
#     ),
#     "answer": "maybe"
#     },
    
#     {
#         "context": "Vitamin C supplements were tested in a randomized trial for cancer prevention. The study found no difference.",
#         "question": "Does Vitamin C prevent cancer?",
#         "rationale": "The trial showed no significant difference between the treatment and control groups, indicating no preventive effect.",
#         "answer": "no"
#     }
# ]

# Prepare a curated 6-shot prompt: 2 yes, 2 no, 2 maybe (with consistent tone and strength)
# from random import sample
import random
from collections import defaultdict

# Group by label
yes_examples = [v for v in train_data.values() if v["final_decision"] == "yes"]
no_examples = [v for v in train_data.values() if v["final_decision"] == "no"]
maybe_examples = [v for v in train_data.values() if v["final_decision"] == "maybe"]

# Sample equal number from each (e.g., 5 each)
sample_size = 3
few_shot_curated = random.sample(yes_examples, sample_size) + \
                   random.sample(no_examples, sample_size) + \
                   random.sample(maybe_examples, sample_size)

# Shuffle to avoid label-order bias
random.shuffle(few_shot_curated)

few_shot_curated = [
    {
        "question": ex["QUESTION"],
        "contexts": ex["CONTEXTS"],
        "long_answer": ex["LONG_ANSWER"],
        "final_decision": ex["final_decision"]
    }
    for ex in few_shot_curated
]


import pandas as pd
import IPython.display as disp
disp.display(pd.DataFrame(few_shot_curated))



,question,contexts,long_answer,final_decision
0,Are there effects of intrauterine cocaine expo...,[To ascertain whether level of intrauterine co...,"In this cohort, prospectively ascertained pren...",no
1,Young-Burgess classification of pelvic ring fr...,[The objectives of this study were to evaluate...,The Young-Burgess system is useful for predict...,maybe
2,Can dose reduction to one parotid gland preven...,[Dryness of the mouth is one of the most distr...,Minimising the radiation dose to one of the pa...,yes
3,Could the extent of lymphadenectomy be modifie...,[The effect of neoadjuvant chemotherapy (NACT)...,The frequency and topographic distribution of ...,no
4,Does anterior laxity of the uninjured knee inf...,[The purpose of this study was to evaluate the...,Greater anterior laxity of the uninjured knee ...,yes
5,Is the international normalised ratio (INR) re...,[As part of an MRC funded study into primary c...,No technical problems associated with INR test...,maybe
6,Maternal creatine homeostasis is altered durin...,[Pregnancy induces adaptations in maternal met...,Change of maternal creatine status (increased ...,yes
7,Is there a role for leukocyte and CRP measurem...,[The diagnosis of acute appendicitis is still ...,Although elevated leukocyte count and CRP valu...,maybe
8,Metered-dose inhalers. Do health care provider...,[The specific aim of this investigation was to...,This study confirms that a large percentage of...,no


# Define the tool

In [11]:
from pydantic import BaseModel
from typing import Literal

class AnswerSchema(BaseModel):
    answer: Literal["yes", "no", "maybe"]

class ReasoningInput(BaseModel):
    question: str
    context: str

In [12]:
from langchain.agents import tool

@tool
def retrieve_pubmed_context(question: str) -> str:
    """Retrieve relevant biomedical abstract context for the given question."""
    # Placeholder logic — replace with real retrieval
    return "Here is some abstract context about the question."

@tool
def summarize_context(context: str) -> str:
    """Summarize the abstract context in one or two sentences."""
    return llm.invoke(f"Summarize this biomedical abstract:\n\n{context}").content.strip()


@tool(args_schema=ReasoningInput)
def evaluate_answer_with_reasoning(input: ReasoningInput) -> str:
    """Use reasoning to answer yes/no/maybe given question and context."""
    prompt = build_prompt({"question": input.question, "context": input.context}, few_shot_curated)
    return llm.invoke(prompt).content.strip()
    

@tool(args_schema=AnswerSchema)
def evaluate_answer_only(answer: str) -> str:
    """Return a normalized yes/no/maybe answer (must be lowercase)."""
    return answer  # Already validated by schema

# Define Prompt Template

In [13]:
def build_prompt(state: dict, few_shot_examples: list) -> str:
    prompt_parts = []

    # Add few-shot reasoning examples
    for ex in few_shot_examples:
        prompt_parts.append(f"Question: {ex['question']}")
        for context in ex['contexts']:
            prompt_parts.append(f"Context: {context}")
        prompt_parts.append(f"Rationale: {ex['long_answer']}")
        prompt_parts.append(f"Answer: {ex['final_decision']}\n")

    # Add the actual user input
    prompt_parts.append(f"Question: {state['question']}")
    prompt_parts.append(f"Context: {state['context']}")

    prompt_parts.append("Rationale:")

    return "\n".join(prompt_parts)


# Create the ReAct agent

In [14]:
from langchain.agents import create_react_agent, AgentExecutor
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate

# 1. Initialize your model
llm = ChatOpenAI(model="gpt-4o")

# 2. Define your tools
tools = [
    retrieve_pubmed_context,
    summarize_context,
    evaluate_answer_with_reasoning,
    evaluate_answer_only,
]

# 3. ReAct-style prompt template (must contain {input}, {agent_scratchpad}, {tools}, {tool_names})
prompt_template = PromptTemplate.from_template("""
You are a biomedical QA agent. Your goal is to answer 'yes', 'no', or 'maybe' based on the provided question and context.

You can use the following tools:
{tools}

When you need to use a tool, use this format:
Thought: I need to use a tool to help answer.
Action: tool_name
Action Input: input to the tool

When you get a result back, use:
Observation: result of the tool

Repeat as needed. When you're ready to answer, say:
Final Answer: yes|no|maybe

Begin!

Question: {input}

{agent_scratchpad}
""")

# 4. Fill in {tools} and {tool_names} using partial
from langchain.agents import Tool

tool_descriptions = "\n".join(f"{tool.name}: {tool.description}" for tool in tools)
tool_names = ", ".join(tool.name for tool in tools)

prompt = prompt_template.partial(
    tools=tool_descriptions,
    tool_names=tool_names,
)

# 5. Create agent and executor
react_agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)
agent_executor = AgentExecutor(agent=react_agent, tools=tools, verbose=True, handle_parsing_errors=True)


In [25]:
question = "Does enteral feeding advancement affect short-term outcomes in very low birth weight infants?"
context = """
    Controversy exists regarding the optimal enteral feeding regimen of very low birth weight infants (VLBW). Rapid advancement of enteral feeding has been associated with an increased rate of necrotizing enterocolitis. In contrast, delaying enteral feeding may have unfavorable effects on nutrition, growth, and neurodevelopment. The aim is to compare the short-term outcomes of VLBW infants in tertiary care centers according to their enteral feeding advancement.",
            "We prospectively studied the influence of center-specific enteral feeding advancement in 1430 VLBW infants recruited from 13 tertiary neonatal intensive care units in Germany on short-term outcome parameters. The centers were post hoc stratified to \"rapid advancement to full enteral feeds\" (median duration of advancement to full enteral feeds<or =12.5 days; 6 centers), that is, rapid advancement (RA), or \"slow advancement to full enteral feeds\" (median duration of advancement to full enteral feeds>12.5 days; 7 centers), that is, slow advancement (SA).",
            "VLBW infants born in centers with SA (n = 713) had a significantly higher rate of sepsis compared with VLBW infants born in centers with RA (n = 717), which was particularly evident for late-onset sepsis (14.0% vs 20.4%; P = 0.002). Furthermore, more central venous lines (48.6% vs 31.1%, P<0.001) and antibiotics (92.4% vs 77.7%, P<0.001) were used in centers with SA."""


input_dict = {
    "input": f"Question: {question}\n\nContext: {context}"
}

response = agent_executor.invoke(input_dict)



In [30]:
response['output']

'Yes'

# Attempting with a minimal working sample

In [26]:
from typing import List
from langchain.agents import AgentExecutor, create_react_agent
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import tool
from langchain.chat_models import ChatOpenAI

# === TOOL DEFINITIONS ===

@tool
def retrieve_pubmed_context(input: str) -> str:
    """Retrieve context from PubMed for a biomedical question."""
    return (
        "Rapid feeding may increase NEC; slow feeding may raise sepsis risk. "
        "The information retrieved suggests that rapid advancement of enteral feeding in Very Low Birth Weight (VLBW) infants "
        "may increase the risk of necrotizing enterocolitis (NEC), a serious intestinal disease. Conversely, advancing feeding "
        "too slowly may increase the risk of sepsis. Therefore, the rate of enteral feeding advancement can significantly affect "
        "health outcomes in VLBW infants by influencing the risks of these complications."
    )

@tool
def classify_answer(input: str) -> str:
    """Classify the answer into 'yes', 'no', or 'maybe' based on the provided explanation."""
    explanation = input.lower()
    if "increase" in explanation and ("nec" in explanation or "sepsis" in explanation):
        return "yes"
    elif "no significant effect" in explanation or "does not affect" in explanation:
        return "no"
    else:
        return "maybe"

tools = [retrieve_pubmed_context, classify_answer]

# === PROMPT ===

prompt = PromptTemplate.from_template(
    """Answer the question using the tools below.

You have access to the following tools:
{tools}

Use the following format:

Question: the input question you must answer  
Thought: you should always think about what to do  
Action: the action to take, should be one of [{tool_names}]  
Action Input: the input to the action  
Observation: the result of the action  
... (this Thought/Action/Action Input/Observation can repeat N times)  
Thought: I now know the final answer  
Final Answer: the final answer to the original input question (only 'yes', 'no', or 'maybe')

Begin!

Question: {input}
{agent_scratchpad}"""
)

# === AGENT ===

llm = ChatOpenAI(model="gpt-4o", temperature=0)
agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)

# === EXAMPLE INVOCATION ===

question = "Does enteral feeding advancement affect outcomes in VLBW infants?"
response = agent_executor.invoke({"input": question})




> Entering new AgentExecutor chain...
Thought: To answer this question, I need to find relevant information from PubMed regarding the impact of enteral feeding advancement on outcomes in very low birth weight (VLBW) infants. 
Action: retrieve_pubmed_context
Action Input: "enteral feeding advancement outcomes VLBW infants"Rapid feeding may increase NEC; slow feeding may raise sepsis risk. The information retrieved suggests that rapid advancement of enteral feeding in Very Low Birth Weight (VLBW) infants may increase the risk of necrotizing enterocolitis (NEC), a serious intestinal disease. Conversely, advancing feeding too slowly may increase the risk of sepsis. Therefore, the rate of enteral feeding advancement can significantly affect health outcomes in VLBW infants by influencing the risks of these complications.The information indicates that the rate of enteral feeding advancement does affect outcomes in VLBW infants, as it influences the risks of necrotizing enterocolitis and sep

In [28]:
response['output']

'yes'

# Attempting with few-shots

In [32]:
from langchain.agents import create_react_agent, AgentExecutor
from langchain.chat_models import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import tool

# === TOOL DEFINITION ===
@tool
def classify_answer(input: str) -> str:
    """Classify biomedical evidence into 'yes', 'no', or 'maybe' based on its implications."""
    explanation = input.lower()
    if "increase" in explanation and ("nec" in explanation or "sepsis" in explanation):
        return "yes"
    elif "no significant effect" in explanation or "does not affect" in explanation:
        return "no"
    else:
        return "maybe"

tools = [classify_answer]

# === PROMPT TEMPLATE ===
prompt = PromptTemplate.from_template(
    """You are a biomedical reasoning assistant.

You have access to the following tools:
{tools}

Use the following format:

Question: the input question you must answer
Thought: think about what you know and what you need to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question (only 'yes', 'no', or 'maybe')

Begin!

Question: {input}
{agent_scratchpad}"""
)

# === AGENT + EXECUTOR ===
llm = ChatOpenAI(model="gpt-4o", temperature=0)
agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)

# === EXAMPLE QUESTION ===
question = (
    "Does enteral feeding advancement affect outcomes in VLBW infants? "
    "Evidence: Rapid feeding may increase NEC; slow feeding may raise sepsis risk. "
    "A recent study found that infants with rapid feed advancement regained birth weight faster, had lower sepsis risk, and shorter hospital stay."
)
response = executor.invoke({"input": question})
response["output"]




> Entering new AgentExecutor chain...
Thought: The question is about whether enteral feeding advancement affects outcomes in very low birth weight (VLBW) infants. The evidence provided suggests that rapid feeding might increase the risk of necrotizing enterocolitis (NEC), while slow feeding might increase the risk of sepsis. However, a recent study indicates that rapid feed advancement is associated with faster weight gain, lower sepsis risk, and shorter hospital stays. I need to classify this evidence to determine the overall implication.

Action: classify_answer
Action Input: Rapid feeding may increase NEC; slow feeding may raise sepsis risk. A recent study found that infants with rapid feed advancement regained birth weight faster, had lower sepsis risk, and shorter hospital stay.yesThe evidence suggests that enteral feeding advancement, specifically rapid feed advancement, positively affects outcomes in VLBW infants by promoting faster weight gain, reducing sepsis risk, and short

'yes'